# Chapter 6. Scenes and Visualization

The code from Chapter 6 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# the central Munich urban scene used in many chapters of this book
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

scene.add(Receiver('rx0', position=[60.0, -40.0, 1.5]))
paths = solver(scene, max_depth=5)

rm = rmsolver(scene, max_depth=3, cell_size=(5.0, 5.0),
              samples_per_tx=10**6)

# receiver trajectory and route length for the animation section
trajectory = lambda t: [10.0 * t - 200.0, 50.0, 1.5]
s = 0.0

## 2. Using a Built-in Scene

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

available = [s for s in dir(scene_assets) if not s.startswith('_')]
print('Available scenes:', available[:10])

scene = load_scene(scene_assets.munich)

## 3. Configuring a Camera

In [ ]:
from sionna.rt import Camera

cam = Camera(position=[-300, 300, 150],
             look_at=[0, 0, 0])

## 4. Previewing a Scene

In [ ]:
scene.preview()

## 5. Rendering from an Oblique Viewpoint

In [ ]:
img = scene.render(camera=cam,
                   resolution=[1280, 720],
                   num_samples=128)

## 6. Displaying Paths

In [ ]:
img = scene.render(camera=cam, paths=paths,
                   resolution=[1280, 720], num_samples=64)

## 7. Overlaying a Radio Map

In [ ]:
img = scene.render(camera=cam, radio_map=rm,
                   resolution=[1280, 720])

## 8. Drawing Results in matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
rss_dbm = 10*np.log10(rm.rss.numpy()[0]) + 30
plt.imshow(rss_dbm, origin='lower', cmap='jet', vmin=-120, vmax=-40)
plt.colorbar(label='RSS [dBm]')
plt.xlabel('x [cells]'); plt.ylabel('y [cells]')

## 9. Making a Readable Figure

In [ ]:
plt.rcParams['font.size'] = 11
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(rss_dbm, origin='lower', cmap='viridis', vmin=-120, vmax=-40)
fig.colorbar(im, ax=ax, label='RSS [dBm]')
ax.set_title('RSS map 3.5 GHz')
plt.tight_layout()
plt.savefig('figs/ch06/fig_ch06_rss_readable.pdf', dpi=300)

## 10. Making an Animation

**Note**: the drawing itself is omitted in the text. `ani.save` requires ffmpeg.

In [ ]:
from matplotlib.animation import FuncAnimation
from sionna.rt import PathSolver
solver = PathSolver()
fig, ax = plt.subplots()
def update(t):
    # move the receiver for time t and recompute the CIR
    rx_pos = trajectory(t)
    scene.remove('rx0')
    scene.add(Receiver('rx0', position=rx_pos))
    paths = solver(scene, max_depth=3)
    ax.cla()
    # plot the CIR here ...
ani = FuncAnimation(fig, update, frames=60, interval=100)
ani.save('figs/ch06/anim_mobile_rx.mp4')